# Baseline Evaluation Analysis — REG² Pathology Report Gen

Presentation-ready figures and diagnostics for thumbnail / RAG / patch-retrieve baselines.

**Key framing for slides:**
- **Chain metrics** (BPV, Edge-F1, MESS) = primary REG² signal — real vision/reasoning difficulty.
- **Report metrics** (ROUGE, BLEU, clinical proxy) = currently **misleading** — structured VLM output vs narrative GT.
- **B1 (HippoRAG2)** = negative RAG ablation (routing collapse to `mass_lesion`).
- **B2 (HybridRAG)** = positive RAG signal on chains (+Edge-F1 vs A).
- **Patch-retrieve** = incomplete (22/69 encoded); **beats thumbnail A on matched cases**.

Run from repo root or `notebooks/` on cluster. Paths from `configs/paths.yaml`.

In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

cfg = yaml.safe_load((REPO / "configs" / "paths.yaml").read_text())
RUNS_DIR = Path(cfg["user"]["work_dir"]) / "runs"
CHAINS_PATH = REPO / "data" / "labels" / "chains.jsonl"
CACHE_ROOT = Path(cfg["user"]["cache_dir"])

BASELINES = {
    "A (flat thumbnail)": RUNS_DIR / "predictions_test_baseline_a.jsonl",
    "B1 (HippoRAG2)": RUNS_DIR / "predictions_test_baseline_b1.jsonl",
    "B2 (HybridRAG)": RUNS_DIR / "predictions_test_baseline_b2.jsonl",
    "Patch (k=100 centroids)": RUNS_DIR / "predictions_test_baseline_patch_retrieve.jsonl",
    "Patch (full pool)": RUNS_DIR / "predictions_test_baseline_patch_retrieve_fullpool.jsonl",
}

sns.set_theme(style="whitegrid", context="talk", font_scale=0.85)
PALETTE = sns.color_palette("Set2", 6)
FIG_DIR = REPO / "notebooks" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("REPO:", REPO)
print("RUNS:", RUNS_DIR)
for name, path in BASELINES.items():
    status = "OK" if path.exists() else "missing"
    n = sum(1 for _ in open(path)) if path.exists() else 0
    print(f"  [{status}] {name}: {path.name} ({n} cases)")

In [ ]:
from eval.edge_parser import chain_dict_to_record
from eval.metrics.chain import binary_path_validity, edge_f1, mess_score
from eval.metrics.report import bleu4, clinical_accuracy_placeholder, rouge_l
from scripts.inference.run_test_baseline_batch import _has_patch_embeddings
from data.case_slides import iter_chain_records, primary_wsi_for_baseline


def load_jsonl(path: Path) -> dict[str, dict]:
    out: dict[str, dict] = {}
    if not path.exists():
        return out
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                rec = json.loads(line)
                out[rec["slide_id"]] = rec
    return out


def node_answers(rec: dict) -> dict[str, str]:
    return {
        (s.get("node_id") or s.get("question", "")): s.get("answer", "")
        for s in rec.get("chain-of-thought", [])
    }


def first_divergence(pred: dict, gt: dict) -> tuple[str | None, str | None, str | None]:
    pa, ga = node_answers(pred), node_answers(gt)
    for nid in gt.get("node_path") or list(ga.keys()):
        if nid in pa and pa[nid] != ga.get(nid):
            return nid, ga.get(nid), pa.get(nid)
        if nid in pa and nid not in ga:
            return nid, None, pa.get(nid)
    for nid in pa:
        if nid not in ga:
            return nid, None, pa[nid]
    return None, None, None


def case_metrics(pred_raw: dict, gt_raw: dict) -> dict:
    p = chain_dict_to_record(pred_raw)
    g = chain_dict_to_record(gt_raw)
    f1 = edge_f1(p, g)
    div_node, div_gt, div_pred = first_divergence(pred_raw, gt_raw)
    return {
        "bpv": binary_path_validity(p, g),
        "edge_f1": f1["f1"],
        "edge_precision": f1["precision"],
        "edge_recall": f1["recall"],
        "mess": mess_score(p, g),
        "rouge_l": rouge_l(p.report, g.report),
        "bleu4": bleu4(p.report, g.report),
        "clinical_proxy": clinical_accuracy_placeholder(p.report, g.report),
        "pred_report_len": len(p.report or ""),
        "gt_report_len": len(g.report or ""),
        "exact_path": pred_raw.get("node_path") == gt_raw.get("node_path"),
        "first_div_node": div_node,
        "first_div_gt": div_gt,
        "first_div_pred": div_pred,
    }


def eval_baseline(name: str, path: Path, gt: dict[str, dict]) -> pd.DataFrame:
    preds = load_jsonl(path)
    rows = []
    for sid in sorted(set(preds) & set(gt)):
        m = case_metrics(preds[sid], gt[sid])
        m.update({"slide_id": sid, "baseline": name, "n_slides": len(sid.split(","))})
        rows.append(m)
    return pd.DataFrame(rows)


gt_all = load_jsonl(CHAINS_PATH)
gt_test = {k: v for k, v in gt_all.items() if v.get("split") == "test"}
print(f"GT test cases: {len(gt_test)}")

frames = []
for name, path in BASELINES.items():
    if path.exists():
        frames.append(eval_baseline(name, path, gt_test))
df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
df.head()

In [ ]:
CHAIN_METRICS = ["bpv", "edge_f1", "mess"]
REPORT_METRICS = ["rouge_l", "bleu4", "clinical_proxy"]
METRIC_LABELS = {
    "bpv": "Binary Path Validity",
    "edge_f1": "Edge-F1",
    "mess": "MESS",
    "rouge_l": "ROUGE-L",
    "bleu4": "BLEU-4",
    "clinical_proxy": "Clinical (token F1)",
}

summary = (
    df.groupby("baseline")[CHAIN_METRICS + REPORT_METRICS + ["pred_report_len", "gt_report_len"]]
    .agg(["mean", "count"])
    .round(4)
)
display(summary)

mean_tbl = df.groupby("baseline")[CHAIN_METRICS + REPORT_METRICS].mean().round(4)
mean_tbl["n"] = df.groupby("baseline").size()
mean_tbl = mean_tbl[["n"] + CHAIN_METRICS + REPORT_METRICS]
display(mean_tbl)

## 1. Chain metrics (primary REG² signal)

These measure whether the agent navigates the diagnostic graph correctly.
GT chains are **text-oracle** (extracted from full reports); predictions use **vision on one WSI**.

In [ ]:
def plot_metric_bars(data: pd.DataFrame, metrics: list[str], title: str, fname: str):
    plot_df = data.groupby("baseline")[metrics].mean().reset_index()
    long = plot_df.melt(id_vars="baseline", var_name="metric", value_name="score")
    long["metric"] = long["metric"].map(METRIC_LABELS)

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=long, x="metric", y="score", hue="baseline", palette=PALETTE, ax=ax)
    ax.set_ylim(0, max(0.5, long["score"].max() * 1.15))
    ax.set_title(title)
    ax.set_ylabel("Score (mean)")
    ax.legend(title="Baseline", bbox_to_anchor=(1.02, 1), loc="upper left")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=2, fontsize=9)
    plt.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=150, bbox_inches="tight")
    plt.show()


if not df.empty:
    plot_metric_bars(
        df,
        CHAIN_METRICS,
        "Chain metrics — test split (higher is better)",
        "chain_metrics_comparison.png",
    )

## 2. Report metrics — interpret with caution

Pred reports are **short structured templates** (~112 chars); GT reports are **full narratives** (~764 chars).
Low ROUGE/BLEU does **not** mean zero clinical utility — see report-length plot below.

In [ ]:
if not df.empty:
    plot_metric_bars(
        df,
        REPORT_METRICS,
        "Report metrics — genre mismatch inflates apparent failure",
        "report_metrics_comparison.png",
    )

## 3. RAG routing analysis — why B1 fails, B2 helps

B1 (HippoRAG2) retrieves **globally across train CoT steps** with no node filter → all cases routed to `mass_lesion`.

In [ ]:
def compartment_counts(preds: dict[str, dict]) -> Counter:
    c = Counter()
    for rec in preds.values():
        for s in rec.get("chain-of-thought", []):
            if s.get("node_id") == "compartment":
                c[s.get("answer", "?")] += 1
    return c


comp_rows = []
for ans, cnt in compartment_counts(gt_test).items():
    comp_rows.append({"baseline": "GT (test)", "compartment": ans, "count": cnt})
for label, path in BASELINES.items():
    if not path.exists():
        continue
    preds = load_jsonl(path)
    for ans, cnt in compartment_counts(preds).items():
        comp_rows.append({"baseline": label, "compartment": ans, "count": cnt})

comp_df = pd.DataFrame(comp_rows)

fig, ax = plt.subplots(figsize=(12, 5))
order = comp_df.groupby("compartment")["count"].sum().sort_values(ascending=False).index
sns.barplot(data=comp_df, x="compartment", y="count", hue="baseline", order=order, ax=ax)
ax.set_title("Predicted compartment distribution vs GT")
ax.tick_params(axis="x", rotation=30)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Source")
plt.tight_layout()
fig.savefig(FIG_DIR / "compartment_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
a_df = df[df["baseline"] == "A (flat thumbnail)"]
if not a_df.empty:
    div_counts = a_df["first_div_node"].value_counts().head(10)
    fig, ax = plt.subplots(figsize=(10, 5))
    div_counts.sort_values().plot(kind="barh", ax=ax, color=PALETTE[0])
    ax.set_title("Baseline A — first graph node where prediction diverges from GT")
    ax.set_xlabel("Cases")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "first_divergence_node_A.png", dpi=150, bbox_inches="tight")
    plt.show()
    display(div_counts.to_frame("count"))

In [ ]:
def organ_procedure_confusion(preds: dict, gt: dict) -> pd.DataFrame:
    rows = []
    for sid in preds:
        if sid not in gt:
            continue
        pa = node_answers(preds[sid]).get("organ_procedure")
        ga = node_answers(gt[sid]).get("organ_procedure")
        if pa and ga:
            rows.append({"gt": ga, "pred": pa})
    return pd.crosstab(pd.Series([r["gt"] for r in rows]), pd.Series([r["pred"] for r in rows]))


a_preds = load_jsonl(BASELINES["A (flat thumbnail)"])
if a_preds:
    ct = organ_procedure_confusion(a_preds, gt_test)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title("organ_procedure confusion — Baseline A\n(curettage→hysterectomy dominates errors)")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("GT")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "organ_procedure_confusion_A.png", dpi=150, bbox_inches="tight")
    plt.show()

## 4. Report length mismatch (why ROUGE is ~0.03)

In [ ]:
if not df.empty:
    len_rows = []
    for _, row in df.iterrows():
        len_rows.append({"baseline": row["baseline"], "length": row["pred_report_len"], "type": "Prediction"})
    for _, row in df.drop_duplicates("slide_id").iterrows():
        len_rows.append({"baseline": "GT", "length": row["gt_report_len"], "type": "Ground truth"})
    len_df = pd.DataFrame(len_rows)

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=len_df, x="type", y="length", hue="baseline", ax=ax)
    ax.set_title("Report length — structured pred vs narrative GT")
    ax.set_ylabel("Characters")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "report_length_boxplot.png", dpi=150, bbox_inches="tight")
    plt.show()

## 5. Fair patch-retrieve comparison (same case IDs)

Do **not** compare patch-retrieve headline metrics (n≈22) against full-set A (n=69).
Always align on overlapping `slide_id`s.

In [ ]:
a_path = BASELINES["A (flat thumbnail)"]
p_path = BASELINES["Patch (k=100 centroids)"]
if a_path.exists() and p_path.exists():
    shared = sorted(set(load_jsonl(a_path)) & set(load_jsonl(p_path)))
    fair = df[df["slide_id"].isin(shared) & df["baseline"].isin(["A (flat thumbnail)", "Patch (k=100 centroids)"])]
    fair_mean = fair.groupby("baseline")[CHAIN_METRICS].mean().round(4)
    display(fair_mean)
    print(f"Shared cases: {len(shared)}")

    pivot = fair.pivot(index="slide_id", columns="baseline", values="edge_f1").dropna()
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(pivot["A (flat thumbnail)"], pivot["Patch (k=100 centroids)"], alpha=0.7, s=60)
    lim = [0, max(pivot.max()) * 1.05 + 0.01]
    ax.plot(lim, lim, "k--", alpha=0.4, label="y=x")
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_xlabel("Edge-F1 — A (thumbnail)")
    ax.set_ylabel("Edge-F1 — Patch retrieve")
    ax.set_title(f"Paired Edge-F1 on {len(pivot)} shared cases")
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIG_DIR / "paired_edge_f1_patch_vs_A.png", dpi=150, bbox_inches="tight")
    plt.show()

## 5b. K-means vs full-pool patch retrieval

Fair comparison on **the same `slide_id`s** — only difference is retrieval pool:
- **k=100 centroids** (default, fast)
- **`--search-all-patches`** (full CONCH pool, slower per node)

Run the full-pool ablation only after the Qwen vLLM server is up (see §10).

In [ ]:
k_path = BASELINES["Patch (k=100 centroids)"]
fp_path = BASELINES["Patch (full pool)"]

if not fp_path.exists():
    print(f"Missing {fp_path.name} — run full-pool ablation first:")
    print("  1) sbatch scripts/cluster/start_qwen_server.sh")
    print("  2) QWEN_API_BASE=http://<qwen-node>:8000/v1 SEARCH_ALL_PATCHES=1 FORCE=1 sbatch scripts/cluster/run_patch_retrieve_baseline.sh")
elif not k_path.exists():
    print(f"Missing {k_path.name}")
else:
    k_preds, fp_preds = load_jsonl(k_path), load_jsonl(fp_path)
    shared = sorted(set(k_preds) & set(fp_preds))
    print(f"Paired cases (k-means ∩ full pool): {len(shared)}")

    pool_df = df[
        df["slide_id"].isin(shared)
        & df["baseline"].isin(["Patch (k=100 centroids)", "Patch (full pool)"])
    ]
    pool_mean = pool_df.groupby("baseline")[CHAIN_METRICS].mean().round(4)
    display(pool_mean)

    pivot = pool_df.pivot(index="slide_id", columns="baseline", values="edge_f1").dropna()
    delta = pivot["Patch (full pool)"] - pivot["Patch (k=100 centroids)"]
    print(
        f"Edge-F1 delta (full − k-means): mean={delta.mean():+.4f}, "
        f"median={delta.median():+.4f}, full wins={(delta > 0).sum()}/{len(delta)}"
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    ax.scatter(
        pivot["Patch (k=100 centroids)"],
        pivot["Patch (full pool)"],
        alpha=0.75,
        s=60,
        c=delta,
        cmap="RdYlGn",
        vmin=-0.2,
        vmax=0.2,
    )
    lim = [0, max(pivot.max()) * 1.05 + 0.01]
    ax.plot(lim, lim, "k--", alpha=0.4, label="y=x")
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_xlabel("Edge-F1 — k=100 centroids")
    ax.set_ylabel("Edge-F1 — full pool")
    ax.set_title(f"Paired Edge-F1 ({len(pivot)} cases)")
    ax.legend()

    ax = axes[1]
    sns.barplot(
        data=pool_mean.reset_index().melt(id_vars="baseline", var_name="metric", value_name="score"),
        x="metric",
        y="score",
        hue="baseline",
        palette=PALETTE[:2],
        ax=ax,
    )
    ax.set_ylim(0, max(0.5, pool_mean.max().max() * 1.15))
    ax.set_title("Mean chain metrics — retrieval pool ablation")
    ax.set_ylabel("Score")
    ax.legend(title="Pool", bbox_to_anchor=(1.02, 1), loc="upper left")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=2, fontsize=9)

    plt.tight_layout()
    fig.savefig(FIG_DIR / "kmeans_vs_fullpool_patch_retrieve.png", dpi=150, bbox_inches="tight")
    plt.show()

## 6. Node-level accuracy heatmap (Baseline A)

In [ ]:
KEY_NODES = [
    "organ_procedure",
    "compartment",
    "endometrium_assessment",
    "diagnosis",
    "mass_histologic_type",
]

if a_preds:
    acc_rows = []
    for nid in KEY_NODES:
        correct = total = 0
        for sid in a_preds:
            if sid not in gt_test:
                continue
            pa = node_answers(a_preds[sid]).get(nid)
            ga = node_answers(gt_test[sid]).get(nid)
            if ga is None:
                continue
            total += 1
            if pa == ga:
                correct += 1
        acc_rows.append({"node": nid, "accuracy": correct / total if total else 0, "n": total})
    acc_df = pd.DataFrame(acc_rows).set_index("node")

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.heatmap(acc_df[["accuracy"]].T, annot=acc_df["n"].values.reshape(1, -1), fmt="d",
                cmap="RdYlGn", vmin=0, vmax=1, ax=ax,
                cbar_kws={"label": "Accuracy"})
    ax.set_title("Node-level exact-match accuracy — Baseline A (annot = n cases with GT node)")
    ax.set_yticklabels(["accuracy"])
    plt.tight_layout()
    fig.savefig(FIG_DIR / "node_accuracy_heatmap_A.png", dpi=150, bbox_inches="tight")
    plt.show()
    display(acc_df)

## 7. Per-case Edge-F1 distribution

In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.violinplot(data=df, x="baseline", y="edge_f1", inner="quartile", palette=PALETTE, ax=ax)
    ax.set_title("Edge-F1 per case — variance across test set")
    ax.tick_params(axis="x", rotation=20)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "edge_f1_violin.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8. Example case — GT vs prediction side-by-side

In [ ]:
def show_case(slide_id: str, pred_path: Path = a_path):
    preds = load_jsonl(pred_path)
    if slide_id not in preds or slide_id not in gt_test:
        print("Case not found")
        return
    p, g = preds[slide_id], gt_test[slide_id]
    print("=" * 60)
    print("slide_id:", slide_id)
    print("GT path:  ", " → ".join(g.get("node_path", [])))
    print("Pred path:", " → ".join(p.get("node_path", [])))
    node, g_ans, p_ans = first_divergence(p, g)
    if node:
        print(f"First divergence @ {node}: GT={g_ans!r}  Pred={p_ans!r}")
    print("\n--- GT report (first 400 chars) ---")
    print(g.get("report", "")[:400])
    print("\n--- Pred report ---")
    print(p.get("report", ""))


if a_preds:
    # pick a case with partial overlap (interesting) and one with zero overlap
    a_case_df = df[df["baseline"] == "A (flat thumbnail)"]
    if not a_case_df.empty:
        mid = a_case_df.sort_values("edge_f1").iloc[len(a_case_df) // 2]["slide_id"]
        show_case(mid)
        worst = a_case_df.sort_values("edge_f1").iloc[0]["slide_id"]
        print("\n\n--- Worst Edge-F1 case ---")
        show_case(worst)

## 9. Patch embedding coverage (why n=22 today)

In [ ]:
coverage_rows = []
for rec in iter_chain_records(CHAINS_PATH, split="test"):
    case_id = rec["slide_id"]
    wsi = primary_wsi_for_baseline(case_id)
    has_emb = _has_patch_embeddings(CACHE_ROOT, wsi)
    coverage_rows.append({
        "slide_id": case_id,
        "primary_wsi": wsi,
        "n_slides": len(case_id.split(",")),
        "has_patch_emb": has_emb,
    })
cov_df = pd.DataFrame(coverage_rows)
n_ok = cov_df["has_patch_emb"].sum()
print(f"Test cases with primary WSI patch embeddings: {n_ok}/{len(cov_df)}")
display(cov_df[~cov_df["has_patch_emb"]].head(10))

fig, ax = plt.subplots(figsize=(5, 4))
cov_df["has_patch_emb"].value_counts().rename({True: "encoded", False: "missing"}).plot(
    kind="bar", ax=ax, color=[PALETTE[2], PALETTE[1]]
)
ax.set_title("Patch embedding coverage — test primary WSIs")
ax.set_xlabel("")
plt.tight_layout()
fig.savefig(FIG_DIR / "patch_embedding_coverage.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Presentation cheat sheet

### Slide narrative
1. **Pipeline works end-to-end** — 69/69 test cases, all baselines + auto-eval.
2. **Chain reasoning is hard** — thumbnail Edge-F1 ~0.27; main failure = `organ_procedure` (curettage vs hysterectomy from one overview image).
3. **RAG is not monolithic** — B1 global CoT retrieval **hurts** (routing collapse); B2 report retrieval **helps** (+0.12 Edge-F1).
4. **Report metrics need redesign** — ROUGE compares template vs narrative; use node-level + diagnosis fields until Phase 2 report LLM.
5. **Patch retrieval promising** — beats thumbnail on n=22 matched cases; finish encoding 47 missing WSIs.
6. **K-means ablation** — §5b compares centroid pool vs full pool (same 22 cases).

### Limitations to mention
- Single primary WSI per multi-slide case (~79% multi-slide)
- GT = text oracle, pred = vision
- B1 = embedding stub, not full HippoRAG KG
- K-means K=100 retrieval pool (ablate with `--search-all-patches`)

### Cluster workflow — Qwen server + full-pool ablation

The baseline script **does not** start Qwen. Phase 1 calls a **separate** vLLM job over HTTP.

```bash
# Step 1 — start Qwen (long-running GPU job; keep it alive)
sbatch scripts/cluster/start_qwen_server.sh

# Step 2 — confirm node + API (replace heidelberg with your node from squeue)
squeue -u $USER
curl -s http://heidelberg:8000/v1/models | python3 -m json.tool | head

# Step 3 — submit full-pool ablation (22 encoded cases, ~hours)
QWEN_API_BASE=http://heidelberg:8000/v1 SEARCH_ALL_PATCHES=1 FORCE=1 \
  sbatch scripts/cluster/run_patch_retrieve_baseline.sh

# Optional smoke first (3 cases):
QWEN_API_BASE=http://heidelberg:8000/v1 SMOKE=1 FORCE=1 \
  sbatch scripts/cluster/run_patch_retrieve_baseline.sh
```

**Cursor SSH:** your SSH session is just the login shell. `sbatch` queues independent SLURM jobs on compute nodes — closing Cursor does not cancel them. You only need SSH to submit/monitor (`squeue`, `tail -f dominik/logs/...`).

Figures saved to `notebooks/figures/` (incl. `kmeans_vs_fullpool_patch_retrieve.png` after full-pool run).